In [1]:
# ========================
# LIMPIEZA COMBINE HISTÓRICO
# ========================

import pandas as pd
import numpy as np

# cargo el dataset del combine
combine = pd.read_csv('../datos/combine_historico.csv')

print(f"Shape: {combine.shape}")
print(f"\nColumnas: {list(combine.columns)}")

Shape: (1873, 47)

Columnas: ['SEASON', 'PLAYER_ID', 'FIRST_NAME', 'LAST_NAME', 'PLAYER_NAME', 'POSITION', 'HEIGHT_WO_SHOES', 'HEIGHT_WO_SHOES_FT_IN', 'HEIGHT_W_SHOES', 'HEIGHT_W_SHOES_FT_IN', 'WEIGHT', 'WINGSPAN', 'WINGSPAN_FT_IN', 'STANDING_REACH', 'STANDING_REACH_FT_IN', 'BODY_FAT_PCT', 'HAND_LENGTH', 'HAND_WIDTH', 'STANDING_VERTICAL_LEAP', 'MAX_VERTICAL_LEAP', 'LANE_AGILITY_TIME', 'MODIFIED_LANE_AGILITY_TIME', 'THREE_QUARTER_SPRINT', 'BENCH_PRESS', 'SPOT_FIFTEEN_CORNER_LEFT', 'SPOT_FIFTEEN_BREAK_LEFT', 'SPOT_FIFTEEN_TOP_KEY', 'SPOT_FIFTEEN_BREAK_RIGHT', 'SPOT_FIFTEEN_CORNER_RIGHT', 'SPOT_COLLEGE_CORNER_LEFT', 'SPOT_COLLEGE_BREAK_LEFT', 'SPOT_COLLEGE_TOP_KEY', 'SPOT_COLLEGE_BREAK_RIGHT', 'SPOT_COLLEGE_CORNER_RIGHT', 'SPOT_NBA_CORNER_LEFT', 'SPOT_NBA_BREAK_LEFT', 'SPOT_NBA_TOP_KEY', 'SPOT_NBA_BREAK_RIGHT', 'SPOT_NBA_CORNER_RIGHT', 'OFF_DRIB_FIFTEEN_BREAK_LEFT', 'OFF_DRIB_FIFTEEN_TOP_KEY', 'OFF_DRIB_FIFTEEN_BREAK_RIGHT', 'OFF_DRIB_COLLEGE_BREAK_LEFT', 'OFF_DRIB_COLLEGE_TOP_KEY', 'OFF_

In [2]:
# ========================
# PRIMERA INSPECCIÓN
# ========================

# veo todas las columnas completas
print("Columnas completas:")
for col in combine.columns:
    print(f"  {col}")

print(f"\nTemporadas disponibles: {sorted(combine['SEASON'].unique())}")
print(f"\nEspañoles en 2026:")
print(combine[combine['SEASON'] == 2026][['PLAYER_NAME', 'POSITION', 'HEIGHT_WO_SHOES', 'WEIGHT', 'WINGSPAN']].to_string())

Columnas completas:
  SEASON
  PLAYER_ID
  FIRST_NAME
  LAST_NAME
  PLAYER_NAME
  POSITION
  HEIGHT_WO_SHOES
  HEIGHT_WO_SHOES_FT_IN
  HEIGHT_W_SHOES
  HEIGHT_W_SHOES_FT_IN
  WEIGHT
  WINGSPAN
  WINGSPAN_FT_IN
  STANDING_REACH
  STANDING_REACH_FT_IN
  BODY_FAT_PCT
  HAND_LENGTH
  HAND_WIDTH
  STANDING_VERTICAL_LEAP
  MAX_VERTICAL_LEAP
  LANE_AGILITY_TIME
  MODIFIED_LANE_AGILITY_TIME
  THREE_QUARTER_SPRINT
  BENCH_PRESS
  SPOT_FIFTEEN_CORNER_LEFT
  SPOT_FIFTEEN_BREAK_LEFT
  SPOT_FIFTEEN_TOP_KEY
  SPOT_FIFTEEN_BREAK_RIGHT
  SPOT_FIFTEEN_CORNER_RIGHT
  SPOT_COLLEGE_CORNER_LEFT
  SPOT_COLLEGE_BREAK_LEFT
  SPOT_COLLEGE_TOP_KEY
  SPOT_COLLEGE_BREAK_RIGHT
  SPOT_COLLEGE_CORNER_RIGHT
  SPOT_NBA_CORNER_LEFT
  SPOT_NBA_BREAK_LEFT
  SPOT_NBA_TOP_KEY
  SPOT_NBA_BREAK_RIGHT
  SPOT_NBA_CORNER_RIGHT
  OFF_DRIB_FIFTEEN_BREAK_LEFT
  OFF_DRIB_FIFTEEN_TOP_KEY
  OFF_DRIB_FIFTEEN_BREAK_RIGHT
  OFF_DRIB_COLLEGE_BREAK_LEFT
  OFF_DRIB_COLLEGE_TOP_KEY
  OFF_DRIB_COLLEGE_BREAK_RIGHT
  ON_MOVE_FIFTEEN
  ON_MOVE_

In [4]:
# ========================
# ESPAÑOLES EN 2026 Y NaN
# ========================

# busco los españoles por nombre
print("=== ESPAÑOLES EN COMBINE 2026 ===")
espanoles = combine[
    (combine['SEASON'] == 2026) &
    (combine['PLAYER_NAME'].str.contains('Mara|Miller|Larrea', case=False, na=False))
]
print(espanoles[['PLAYER_NAME', 'POSITION', 'HEIGHT_WO_SHOES', 'WEIGHT', 'WINGSPAN', 'STANDING_REACH', 'MAX_VERTICAL_LEAP', 'LANE_AGILITY_TIME']].to_string())

# NaN por columna
print("\n=== NaN POR COLUMNA (%) ===")
miss = (combine.isnull().sum() / len(combine) * 100).sort_values(ascending=False)
print(miss[miss > 0].round(2).to_string())

=== ESPAÑOLES EN COMBINE 2026 ===
           PLAYER_NAME POSITION  HEIGHT_WO_SHOES  WEIGHT  WINGSPAN  STANDING_REACH  MAX_VERTICAL_LEAP  LANE_AGILITY_TIME
1818  Sergio De Larrea       PG              NaN     NaN       NaN             NaN                NaN                NaN
1837         Aday Mara        C             87.0   259.8     90.00           117.0               28.0              11.47
1839       Baba Miller       PF             82.5   208.2     85.75           111.0               34.5              10.71

=== NaN POR COLUMNA (%) ===
OFF_DRIB_COLLEGE_BREAK_RIGHT    98.34
OFF_DRIB_COLLEGE_TOP_KEY        98.34
SPOT_FIFTEEN_CORNER_LEFT        94.45
SPOT_FIFTEEN_BREAK_RIGHT        94.34
SPOT_FIFTEEN_BREAK_LEFT         94.34
SPOT_FIFTEEN_TOP_KEY            94.34
SPOT_FIFTEEN_CORNER_RIGHT       94.34
SPOT_COLLEGE_BREAK_LEFT         90.87
SPOT_COLLEGE_CORNER_RIGHT       90.87
SPOT_COLLEGE_TOP_KEY            90.87
SPOT_COLLEGE_BREAK_RIGHT        90.87
ON_MOVE_FIFTEEN                 90.

In [5]:
# ========================
# LIMPIEZA DEL COMBINE
# ========================

# elimino columnas con más del 50% de NaN (columnas de tiro, casi vacías)
cols_eliminar = [col for col in combine.columns if combine[col].isnull().mean() > 0.5]
combine = combine.drop(columns=cols_eliminar)
print(f"Columnas eliminadas por >50% NaN: {len(cols_eliminar)}")
print(f"Columnas restantes: {combine.shape[1]}")

# elimino también las columnas en formato pies/pulgadas (redundantes con las numéricas)
cols_ft_in = [col for col in combine.columns if 'FT_IN' in col]
combine = combine.drop(columns=cols_ft_in)
print(f"Columnas FT_IN eliminadas: {cols_ft_in}")

# elimino PLAYER_ID (identificador interno, no aporta)
combine = combine.drop(columns=['PLAYER_ID'])

# compruebo lo que queda
print(f"\nColumnas finales: {list(combine.columns)}")

Columnas eliminadas por >50% NaN: 24
Columnas restantes: 23
Columnas FT_IN eliminadas: ['HEIGHT_WO_SHOES_FT_IN', 'HEIGHT_W_SHOES_FT_IN', 'WINGSPAN_FT_IN', 'STANDING_REACH_FT_IN']

Columnas finales: ['SEASON', 'FIRST_NAME', 'LAST_NAME', 'PLAYER_NAME', 'POSITION', 'HEIGHT_WO_SHOES', 'HEIGHT_W_SHOES', 'WEIGHT', 'WINGSPAN', 'STANDING_REACH', 'BODY_FAT_PCT', 'HAND_LENGTH', 'HAND_WIDTH', 'STANDING_VERTICAL_LEAP', 'MAX_VERTICAL_LEAP', 'LANE_AGILITY_TIME', 'THREE_QUARTER_SPRINT', 'BENCH_PRESS']


In [6]:
# ========================
# IMPUTACIÓN DE NaN Y SERGIO DE LARREA
# ========================

# columnas físicas numéricas que quedan
cols_fisicas = ['HEIGHT_WO_SHOES', 'HEIGHT_W_SHOES', 'WEIGHT', 'WINGSPAN',
                'STANDING_REACH', 'BODY_FAT_PCT', 'HAND_LENGTH', 'HAND_WIDTH',
                'STANDING_VERTICAL_LEAP', 'MAX_VERTICAL_LEAP',
                'LANE_AGILITY_TIME', 'MODIFIED_LANE_AGILITY_TIME',
                'THREE_QUARTER_SPRINT', 'BENCH_PRESS']

# convierto a numérico por si hay algún valor corrupto
for col in cols_fisicas:
    if col in combine.columns:
        combine[col] = pd.to_numeric(combine[col], errors='coerce')

# imputo NaN con la mediana de cada columna
for col in cols_fisicas:
    if col in combine.columns:
        combine[col] = combine[col].fillna(combine[col].median())

# imputo POSITION con la moda
combine['POSITION'] = combine['POSITION'].fillna(combine['POSITION'].mode()[0])

# corrijo a Sergio De Larrea con sus datos reales (fuentes externas)
# altura sin zapatillas: 198cm = 77.95 pulgadas
# peso: 174 lbs (79kg), envergadura: 202cm = 79.53 pulgadas
# alcance de pie: estimado ~255cm = 100.4 pulgadas
mask_sergio = combine['PLAYER_NAME'] == 'Sergio De Larrea'
combine.loc[mask_sergio, 'HEIGHT_WO_SHOES'] = 77.95
combine.loc[mask_sergio, 'WEIGHT']          = 174.2
combine.loc[mask_sergio, 'WINGSPAN']        = 79.53
combine.loc[mask_sergio, 'STANDING_REACH']  = 100.4

# compruebo que Sergio queda bien
print("=== SERGIO DE LARREA CORREGIDO ===")
print(combine[mask_sergio][['PLAYER_NAME','HEIGHT_WO_SHOES','WEIGHT','WINGSPAN','STANDING_REACH']].to_string())

# compruebo NaN restantes
print(f"\nNaN restantes: {combine.isnull().sum().sum()}")
print(f"\nShape final: {combine.shape}")

=== SERGIO DE LARREA CORREGIDO ===
           PLAYER_NAME  HEIGHT_WO_SHOES  WEIGHT  WINGSPAN  STANDING_REACH
1818  Sergio De Larrea            77.95   174.2     79.53           100.4

NaN restantes: 0

Shape final: (1873, 18)


In [7]:
# ========================
# GUARDAR COMBINE LIMPIO
# ========================

# guardo el combine limpio en la carpeta de datos procesados
combine.to_csv('../datos/procesados/combine_final.csv', index=False)

# comprobación final con los tres españoles
print("=== DATASET COMBINE FINAL ===")
print(f"Shape: {combine.shape}")
print(f"\nLos tres españoles:")
espanoles = combine[combine['PLAYER_NAME'].isin(['Aday Mara', 'Baba Miller', 'Sergio De Larrea'])]
print(espanoles[['PLAYER_NAME', 'POSITION', 'HEIGHT_WO_SHOES', 'WEIGHT', 'WINGSPAN', 'STANDING_REACH', 'MAX_VERTICAL_LEAP', 'LANE_AGILITY_TIME']].to_string())

=== DATASET COMBINE FINAL ===
Shape: (1873, 18)

Los tres españoles:
           PLAYER_NAME POSITION  HEIGHT_WO_SHOES  WEIGHT  WINGSPAN  STANDING_REACH  MAX_VERTICAL_LEAP  LANE_AGILITY_TIME
1818  Sergio De Larrea       PG            77.95   174.2     79.53           100.4               34.5              11.30
1837         Aday Mara        C            87.00   259.8     90.00           117.0               28.0              11.47
1839       Baba Miller       PF            82.50   208.2     85.75           111.0               34.5              10.71
